# 06_05_正弦波_学習型_RoPE_の比較

コード：`ch06/blocks/06_05_正弦波_学習型_RoPE_の比較.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 位置符号化：正弦波（足す）・学習型（表を引く）・RoPE（回す）を比べる
import numpy as np

d, base, L = 8, 10000.0, 24                   # ヘッド次元・周期の基数・学習時の最大長
omega = base ** (-np.arange(0, d, 2) / d)     # ペアごとの角速度 θᵢ = base^(−2i/d)

def sinusoidal(pos):                          # 2017 年の正弦波 PE（sin と cos を交互に）
    pe = np.empty(d)
    pe[0::2], pe[1::2] = np.sin(pos * omega), np.cos(pos * omega)
    return pe

learned = np.random.default_rng(1).normal(0, 0.5, (L, d))   # 学習型：L 行の表

def rope(v, pos):                             # 2 次元ずつペアにして角度 pos·θᵢ だけ回す
    c, s = np.cos(pos * omega), np.sin(pos * omega)
    out = np.empty(d)
    out[0::2] = v[0::2] * c - v[1::2] * s
    out[1::2] = v[0::2] * s + v[1::2] * c
    return out

rng = np.random.default_rng(0)
q, k = rng.normal(size=d), rng.normal(size=d)

def score(method, m, n):
    if method == "正弦波":
        return (q + sinusoidal(m)) @ (k + sinusoidal(n))
    if method == "学習型":
        if n >= L:
            return None                       # 表に無い位置は作れない
        return (q + learned[m]) @ (k + learned[n])
    return rope(q, m) @ rope(k, n)

gap = 5                                       # 相対距離 n − m を固定して、位置だけずらす
for method in ("正弦波", "学習型", "RoPE"):
    vals = [score(method, m, m + gap) for m in range(0, 26)]
    ok = [v for v in vals if v is not None]
    first_none = next((m + gap for m, v in enumerate(vals) if v is None), "-")
    pick = "  ".join(f"m={m}:{vals[m]:6.3f}" if vals[m] is not None else f"m={m}:  なし"
                     for m in (0, 2, 7, 12, 18, 19))
    print(f"{method:4s} {pick}   変動幅={max(ok) - min(ok):.3f}   使えなくなる位置 n={first_none}")

# ベクトルの長さの変化 |q′| / |q|
for method, f in (("正弦波", lambda m: q + sinusoidal(m)), ("RoPE", lambda m: rope(q, m))):
    r = [np.linalg.norm(f(m)) / np.linalg.norm(q) for m in range(0, 26)]
    print(f"{method:4s} |q′|/|q| の範囲 = {min(r):.3f} 〜 {max(r):.3f}")

# 正弦波の 4 項分解（n − m = 5）：位置に依存するのはどの項か
print("m    q·k    PE·PE   q·PE(n)  PE(m)·k")
for m in (0, 2, 7, 12):
    n = m + gap
    print(f"{m:2d}  {q @ k:6.3f}  {sinusoidal(m) @ sinusoidal(n):6.3f}  "
          f"{q @ sinusoidal(n):7.3f}  {sinusoidal(m) @ k:7.3f}")
